In [2]:
import gzip, io
import numpy as np
import pandas as pd

SERIES_MATRIX = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_series_matrix.txt.gz"   # edit this

opener = gzip.open if SERIES_MATRIX.endswith(".gz") else open
meta, table_lines, in_table = {}, [], False

with opener(SERIES_MATRIX, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\n")
        if line.startswith("!series_matrix_table_begin"):
            in_table = True
            continue
        if line.startswith("!series_matrix_table_end"):
            in_table = False
            continue
        if in_table:
            table_lines.append(line)
        elif line.startswith("!Sample_"):
            parts = line.split("\t")
            meta.setdefault(parts[0], []).append([p.strip('"') for p in parts[1:]])

# expression: probes x samples
expr = pd.read_csv(io.StringIO("\n".join(table_lines)), sep="\t", index_col=0)
expr.index = expr.index.astype(str).str.strip('"')
expr.columns = [c.strip('"') for c in expr.columns]

# clinical: one row per sample
gsm = meta["!Sample_geo_accession"][0]
clin = pd.DataFrame({"GSM": gsm, "title": meta["!Sample_title"][0]}, index=gsm)

for row in meta.get("!Sample_characteristics_ch1", []):
    pairs = [v.split(": ", 1) if ": " in v else ["", v] for v in row]
    keys = [p[0] for p in pairs if p[0]]
    if not keys:
        continue
    key = pd.Series(keys).mode()[0]
    clin[key] = [p[1] if p[0] == key else np.nan for p in pairs]

print("Probes x samples:", expr.shape)
print("Value range:", float(np.nanmin(expr.values)), float(np.nanmax(expr.values)))
print("Median of sample medians:", float(expr.median(axis=0).median()))
print("Any NA:", bool(expr.isna().any().any()))
print("Platform:", meta["!Sample_platform_id"][0][0], "\n")
print("Clinical shape:", clin.shape)
print(list(clin.columns), "\n")
print(clin.head())

for c in clin.columns[2:]:
    print("\n--", c)
    print(clin[c].value_counts(dropna=False).head(6).to_string())

expr.to_pickle("GSE62254_expr.pkl")
clin.to_csv("GSE62254_clinical.csv")

Probes x samples: (54675, 300)
Value range: 0.572611153 4.412371159
Median of sample medians: 1.4759190675
Any NA: False
Platform: GPL570 

Clinical shape: (300, 4)
['GSM', 'title', 'tissue', 'patient'] 

                   GSM title         tissue patient
GSM1523727  GSM1523727  T107  Gastric tumor     107
GSM1523728  GSM1523728  T108  Gastric tumor     108
GSM1523729  GSM1523729  T109  Gastric tumor     109
GSM1523744  GSM1523744  T160  Gastric tumor     160
GSM1523745  GSM1523745  T164  Gastric tumor     164

-- tissue
tissue
Gastric tumor    300

-- patient
patient
107    1
108    1
109    1
160    1
164    1
165    1


In [5]:
import gzip

SOFT = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_family\GSE62254_family.soft"   # edit this

with open(SOFT, "rb") as fh:
    magic = fh.read(4)

if magic[:2] == b"\x1f\x8b":
    print("Format: gzip")
    stream = gzip.open(SOFT, "rt", encoding="utf-8", errors="replace")
elif magic[:2] == b"PK":
    print("Format: zip")
    z = zipfile.ZipFile(SOFT)
    print("Files inside:", z.namelist())
    stream = io.TextIOWrapper(z.open(z.namelist()[0]), encoding="utf-8", errors="replace")
else:
    print("Format: plain text")
    stream = open(SOFT, "rt", encoding="utf-8", errors="replace")

n_samples = 0
with stream as f:
    for line in f:
        if line.startswith("^SAMPLE"):
            n_samples += 1
            if n_samples == 2:
                break
            print("\n", line.strip())
            continue
        if n_samples == 1:
            if line.startswith("!sample_table_begin"):
                continue
            if line.startswith("!Sample_"):
                print(line.strip()[:250])

Format: plain text

 ^SAMPLE = GSM1523727
!Sample_title = T107
!Sample_geo_accession = GSM1523727
!Sample_status = Public on Apr 17 2015
!Sample_submission_date = Oct 10 2014
!Sample_last_update_date = Apr 17 2015
!Sample_type = RNA
!Sample_channel_count = 1
!Sample_source_name_ch1 = Tumor from patient 107
!Sample_organism_ch1 = Homo sapiens
!Sample_taxid_ch1 = 9606
!Sample_characteristics_ch1 = tissue: Gastric tumor
!Sample_characteristics_ch1 = patient: 107
!Sample_molecule_ch1 = total RNA
!Sample_extract_protocol_ch1 = Samples are processed in parallel in 96-well plates to minimize potential variation. Reaction purification is achieved using magnetic binding beads for cDNA and Qiagen RNeasy kits for cRNA purification.
!Sample_label_ch1 = biotin
!Sample_label_protocol_ch1 = standard Affymetrix protocol
!Sample_hyb_protocol = GeneChip microarrays are loaded with the fragmented target sample/hybridization buffer mix using standard manual techniques. Arrays are hybridized for 18 hours a

In [7]:
import io, pandas as pd, numpy as np

SOFT = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_family\GSE62254_family.soft"   # same file as before

lines, in_tab = [], False
with open(SOFT, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("!platform_table_begin"):
            in_tab = True
            continue
        if line.startswith("!platform_table_end"):
            break
        if in_tab:
            lines.append(line)
        elif line.startswith("^SAMPLE"):
            break

if not lines:
    print("No platform table in this file. Download the annotation table from the GPL570 page instead.")
else:
    annot = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
    print(annot.shape)
    print(list(annot.columns)[:12])
    annot.to_csv("GPL570_annotation.csv", index=False)

    # scale check: which probes are highest, and what would they be on log2?
    top = expr.mean(axis=1).sort_values(ascending=False).head(8)
    sym = annot.set_index("ID")["Gene Symbol"] if "Gene Symbol" in annot.columns else None
    out = pd.DataFrame({"mean_in_matrix": top.round(3),
                        "if_log10_to_log2": (top * np.log2(10)).round(2)})
    if sym is not None:
        out["gene"] = sym.reindex(out.index).values
    print(out)

(54675, 16)
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID']
                 mean_in_matrix  if_log10_to_log2                   gene
ID_REF                                                                  
1553588_at                4.281             14.22        ND3 /// SH3KBP1
201492_s_at               4.246             14.11                  RPL41
204892_x_at               4.223             14.03                 EEF1A1
1553570_x_at              4.214             14.00                   COX2
212284_x_at               4.207             13.98  LOC101928826 /// TPT1
AFFX-hum_alu_at           4.203             13.96                    NaN
207783_x_at               4.202             13.96                  HUWE1
212869_x_at               4.200             13.95  LOC101928826 /// TPT1


In [8]:
import gzip, numpy as np, pandas as pd

M66229 = r"C:\Users\arz_2\Basira_Motassim_Group_A\validation_data\GSE62254_series_matrix.txt.gz"   # edit this

def read_clin(path):
    opn = gzip.open if path.endswith(".gz") else open
    meta = {}
    with opn(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("!series_matrix_table_begin"):
                break
            if line.startswith("!Sample_"):
                parts = line.rstrip("\n").split("\t")
                meta.setdefault(parts[0], []).append([p.strip('"') for p in parts[1:]])
    gsm = meta["!Sample_geo_accession"][0]
    df = pd.DataFrame({"GSM": gsm, "title": meta["!Sample_title"][0]}, index=gsm)
    for row in meta.get("!Sample_characteristics_ch1", []):
        pairs = [v.split(": ", 1) if ": " in v else ["", v] for v in row]
        keys = [p[0] for p in pairs if p[0]]
        if not keys:
            continue
        key = pd.Series(keys).mode()[0]
        df[key] = [p[1] if p[0] == key else np.nan for p in pairs]
    return df

clin2 = read_clin(M66229)
print(clin2.shape)
print(list(clin2.columns))
print("GSMs shared with our 300:", len(set(clin2.index) & set(clin.index)))

for c in clin2.columns[2:]:
    print("\n--", c)
    print(clin2[c].value_counts(dropna=False).head(8).to_string())

(300, 4)
['GSM', 'title', 'tissue', 'patient']
GSMs shared with our 300: 300

-- tissue
tissue
Gastric tumor    300

-- patient
patient
107    1
108    1
109    1
160    1
164    1
165    1
168    1
169    1


In [9]:
import numpy as np, pandas as pd

# scale: matrix appears to be log10(RMA); convert to log2 (inference, see check below)
expr_log2 = expr * np.log2(10)
print("log2-scale range:", round(float(expr_log2.values.min()), 2), "to", round(float(expr_log2.values.max()), 2))

annot = pd.read_csv("GPL570_annotation.csv", dtype=str).set_index("ID")
sym = annot["Gene Symbol"].reindex(expr_log2.index)

is_multi = sym.str.contains("///", na=False)
is_none  = sym.isna()
is_affx  = pd.Series(expr_log2.index.str.startswith("AFFX"), index=expr_log2.index)
keep = ~(is_multi | is_none | is_affx)
print(f"probes {len(sym)} | no symbol {is_none.sum()} | multi-gene {is_multi.sum()} | control {is_affx.sum()} | kept {keep.sum()}")

# rule: one probe per gene, the one with the highest mean expression
e = expr_log2.loc[keep].copy()
m = e.mean(axis=1).values
e["gene"] = sym[keep].str.strip().values
e = e.assign(_m=m).sort_values("_m", ascending=False).drop_duplicates("gene")
genes = e.set_index("gene").drop(columns="_m")
print("genes x samples:", genes.shape)

CAD = ["CDH1","CDH2","CDH3","CDH11","CDH13","CDH17","CDHR2","CDHR5",
       "CLSTN1","DSC2","DSG2","FAT1","PCDH1","PCDH17","PCDH18","PCDHGA10"]
need = ["TP53", "GZMA", "PRF1"] + CAD
print({g: (g in genes.index) for g in need})

genes.to_csv("ACRG_log2_genes.tsv", sep="\t")

log2-scale range: 1.9 to 14.66
probes 54675 | no symbol 8893 | multi-gene 2796 | control 62 | kept 42976
genes x samples: (21655, 300)
{'TP53': True, 'GZMA': True, 'PRF1': True, 'CDH1': True, 'CDH2': True, 'CDH3': True, 'CDH11': True, 'CDH13': True, 'CDH17': True, 'CDHR2': True, 'CDHR5': True, 'CLSTN1': True, 'DSC2': True, 'DSG2': True, 'FAT1': True, 'PCDH1': True, 'PCDH17': True, 'PCDH18': True, 'PCDHGA10': True}
